# Tweet scoring pipeline

Hi Angelica — I have kept this version as simple as possible so it is easy to follow and easy to modify.

The notebook does the following:

- connects to the Ollama model running on the GPU;
- finds the most similar human-labelled tweets and uses them as examples;
- produces the three scores from 0 to 100;
- keeps the human score when a tweet has already been annotated;
- saves each result straight away, so a crash does not wipe the run;
- resumes from the existing CSV when you restart it;
- creates a separate output file for each experiment setup.

The overall flow is simply:

**choose the settings → load the data → find examples → score each tweet → save the result**


## 1. Start Ollama on the GPU

Run this in a separate terminal before you start the notebook:

```bash
conda activate Enron
module load test-modules singularity

WORK="$HOME/ollama_runtime_0.30.8"

singularity exec --nv   -B "$WORK/root/bin/ollama:/usr/bin/ollama"   -B "$WORK/root/lib/ollama:/usr/lib/ollama"   "$HOME/ollama_latest.sif"   env OLLAMA_HOST=127.0.0.1:11434   /usr/bin/ollama serve
```

Just leave that terminal open while the notebook is running.


In [ ]:
# 2. Import everything we need

import csv
import hashlib
import importlib.metadata
import json
import os
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_ollama import ChatOllama


If Python says that one of the packages is missing, run this once:

```bash
pip install -U langchain langchain-core langchain-ollama pydantic scikit-learn pandas tqdm
```


In [ ]:
# 3. Settings for this experiment

# -----------------------------------------------------------------------------
# These are the three values you will usually change between runs.
# -----------------------------------------------------------------------------

# How many human-labelled tweets should the model see for each new tweet?
# 0 means no examples. We are currently using 6.
NUMBER_OF_EXAMPLES = 6

# This is the total space available for the instructions, examples, tweet and answer.
NUM_CTX = 16_384

# This limits how long the model can make its answer for one tweet.
# It can still stop earlier, which it normally should.
NUM_PREDICT = 700


# Ollama and model settings
MODEL_NAME = os.getenv("OLLAMA_MODEL", "deepseek-r1:8b")
OLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
TEMPERATURE = 0
KEEP_ALIVE = "30m"

# Prompt and example-selection settings
PROMPT_VERSION = "trump-risk-v2.2-grounded-crashsafe"
EXAMPLE_ORDER = "least_to_most_similar"
MAX_RETRIES = 3

# Settings for the full run
PRESERVE_HUMAN_LABELS = True
MAX_TWEETS = None          # None runs everything. Put 10 here for a quick test.
TWEET_SNIPPET_LENGTH = 280
SEED = 42


# Where the input files are and where the results will go
DATA_FOLDER_OVERRIDE = os.getenv("TRUMP_DATA_DIR")
OUTPUT_FOLDER = Path("outputs/llm_annotations")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


def safe_filename_text(value):
    """Make a version of the model name that is safe to use in a filename."""
    return re.sub(r"[^A-Za-z0-9._-]+", "-", str(value)).strip("-")


# I include the settings in the filename so each experiment stays separate
# and a new setup cannot accidentally continue an older run.
MODEL_TAG = safe_filename_text(MODEL_NAME)
RUN_TAG = (
    f"{MODEL_TAG}_fewshot{NUMBER_OF_EXAMPLES}"
    f"_predict{NUM_PREDICT}_ctx{NUM_CTX}"
)

LIVE_CSV = OUTPUT_FOLDER / f"LLM_annotations_{RUN_TAG}.csv"
ERROR_CSV = OUTPUT_FOLDER / f"LLM_annotation_errors_{RUN_TAG}.csv"
MANIFEST_FILE = OUTPUT_FOLDER / f"run_manifest_{RUN_TAG}.json"
MASTER_ANNOTATIONS_CSV = OUTPUT_FOLDER / "master_human_annotations.csv"

FILE_NAMES = {
    "tweets": "cleaned_tweets.csv",
    "human": "tweets_human_annotations.csv",
    "trade": "trade_annotations.csv",
    "sanctions": "sanctions_annotations.csv",
    "fed": "fed_annotations.csv",
}

print("Model:", MODEL_NAME)
print("Few-shot examples:", NUMBER_OF_EXAMPLES)
print("Context length:", NUM_CTX)
print("Maximum generated tokens:", NUM_PREDICT)
print("Maximum tweets:", "ALL" if MAX_TWEETS is None else MAX_TWEETS)
print("Run tag:", RUN_TAG)
print("Output:", LIVE_CSV.resolve())


## Choosing the experiment settings

These are the only three values you will normally need to change:

```python
NUMBER_OF_EXAMPLES = 6
NUM_CTX = 16_384
NUM_PREDICT = 700
```

For the first comparison, I would try:

- **No examples:** `NUMBER_OF_EXAMPLES = 0`
- **A small number of examples:** `NUMBER_OF_EXAMPLES = 3`
- **Our current setup:** `NUMBER_OF_EXAMPLES = 6`
- **A larger set of examples:** `NUMBER_OF_EXAMPLES = 10`

Keep `NUM_CTX = 16_384` and `NUM_PREDICT = 700` at first. This way, the only thing changing between runs is the number of examples, so the comparison is cleaner.

If DeepSeek regularly runs out of tokens before giving the final JSON, we can test `NUM_PREDICT = 1_200` separately.

Each setup gets its own CSV automatically. For example:

```text
LLM_annotations_deepseek-r1-8b_fewshot6_predict700_ctx16384.csv
```


In [ ]:
# 4. A few small functions for loading and cleaning the data

def find_file(filename):
    """Look for the CSV in the usual folders."""
    possible_folders = [
        Path(DATA_FOLDER_OVERRIDE).expanduser() if DATA_FOLDER_OVERRIDE else None,
        Path.cwd(),
        Path.cwd() / "data" / "processed",
        Path.cwd().parent / "data" / "processed",
        Path("/mnt/data"),
    ]

    checked_paths = []
    for folder in possible_folders:
        if folder is None:
            continue
        path = (folder / filename).resolve()
        checked_paths.append(str(path))
        if path.exists():
            return path

    checked = "\n- ".join(checked_paths)
    raise FileNotFoundError(
        f"Could not find {filename!r}. Checked:\n- {checked}\n"
        "Place the CSV beside the notebook, under data/processed, "
        "or set the TRUMP_DATA_DIR environment variable."
    )


def load_csv(path):
    """Load everything as text so Python does not alter the long tweet IDs."""
    data = pd.read_csv(path, dtype=str)
    data.columns = [str(column).strip() for column in data.columns]
    return data


def normalise_text(value):
    """Clean the tweet text so matching is consistent."""
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.replace("&amp;", "&")
    return re.sub(r"\s+", " ", text).strip().lower()


def normalise_date(value):
    """Put every date into the same format."""
    date = pd.to_datetime(value, errors="coerce")
    if pd.isna(date):
        return ""
    return date.strftime("%Y-%m-%d %H:%M:%S")


def make_annotation_key(date, tweet):
    """Create a matching key from the date and tweet text instead of relying on the Tweet ID."""
    value = f"{normalise_date(date)}||{normalise_text(tweet)}"
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def add_annotation_key(data):
    """Add that matching key to the data."""
    output = data.copy()
    output["annotation_key"] = [
        make_annotation_key(date, tweet)
        for date, tweet in zip(output["Date"], output["Tweet"])
    ]
    return output


def file_sha256(path):
    """Record which exact version of each input file was used."""
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def score_band(score):
    """Turn the numerical score into its matching severity band."""
    score = int(score)
    if score == 0:
        return "zero"
    if score <= 19:
        return "very_low"
    if score <= 39:
        return "low"
    if score <= 59:
        return "moderate"
    if score <= 79:
        return "severe"
    return "very_severe"


In [ ]:
# 5. Load the five CSV files and put the annotations together

PATHS = {name: find_file(filename) for name, filename in FILE_NAMES.items()}

for name, path in PATHS.items():
    print(f"{name:10s} -> {path}")

tweets = add_annotation_key(load_csv(PATHS["tweets"]))
human_annotations = add_annotation_key(load_csv(PATHS["human"]))
trade_annotations = add_annotation_key(load_csv(PATHS["trade"]))
sanctions_annotations = add_annotation_key(load_csv(PATHS["sanctions"]))
fed_annotations = add_annotation_key(load_csv(PATHS["fed"]))

# This row number lets us restart the run without scoring the same tweet twice.
tweets = tweets.reset_index(drop=True)
tweets.insert(0, "tweet_index", np.arange(len(tweets), dtype=int))

# A couple of quick checks before we start
for name, data in {
    "tweets": tweets,
    "human": human_annotations,
    "trade": trade_annotations,
    "sanctions": sanctions_annotations,
    "fed": fed_annotations,
}.items():
    missing = {"Tweet", "Date"} - set(data.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {sorted(missing)}")

# Give the three scores the same names everywhere in the notebook.
master_annotations = human_annotations.rename(
    columns={
        "Trade Hostility": "trade_score",
        "Sanction Threat": "sanctions_score",
        "Fed Pressure": "fed_pressure_score",
        "Reason": "human_reason",
    }
).copy()

SCORE_COLUMNS = ["trade_score", "sanctions_score", "fed_pressure_score"]

for column in SCORE_COLUMNS:
    master_annotations[column] = pd.to_numeric(
        master_annotations[column], errors="raise"
    ).astype(int)

    if not master_annotations[column].between(0, 100).all():
        raise ValueError(f"{column} contains a score outside 0-100.")


def prepare_detailed_annotations(data, prefix, score_column):
    """Keep the extra annotation details and give them consistent names."""
    columns = [
        "annotation_key", "Severity", "Role", "Status",
        score_column, "Band", "Reason",
    ]
    output = data[columns].copy()

    return output.rename(
        columns={
            "Severity": f"{prefix}_severity",
            "Role": f"{prefix}_role",
            "Status": f"{prefix}_status",
            score_column: f"{prefix}_detail_score",
            "Band": f"{prefix}_band",
            "Reason": f"{prefix}_reason",
        }
    )


master_annotations = master_annotations.merge(
    prepare_detailed_annotations(
        trade_annotations, "trade", "Trade Score (0-100)"
    ),
    on="annotation_key",
    how="left",
)

master_annotations = master_annotations.merge(
    prepare_detailed_annotations(
        sanctions_annotations, "sanctions", "Sanctions Score (0-100)"
    ),
    on="annotation_key",
    how="left",
)

master_annotations = master_annotations.merge(
    prepare_detailed_annotations(
        fed_annotations, "fed", "Fed Score (0-100)"
    ),
    on="annotation_key",
    how="left",
)

master_annotations.insert(
    0,
    "annotation_id",
    [f"A{number:03d}" for number in range(1, len(master_annotations) + 1)],
)

if master_annotations["annotation_key"].duplicated().any():
    raise ValueError("Duplicate human annotation keys were found.")

master_annotations.to_csv(MASTER_ANNOTATIONS_CSV, index=False)

matched = master_annotations["annotation_key"].isin(set(tweets["annotation_key"])).sum()

print("Human annotations:", len(master_annotations))
print("Tweets to score:", len(tweets))
print(f"Annotations matched to cleaned tweets: {matched}/{len(master_annotations)}")
display(master_annotations.head(3))


### Why I am not matching the files using Tweet ID

Some of the tweet IDs appear in scientific notation, which can remove the last few digits. To avoid matching the wrong tweets, I use the date and the tweet text together instead.

The original Tweet ID is still kept in the final output.


In [ ]:
# 6. Find the human examples that are closest to each tweet

def build_reference_bank(annotations):
    """Prepare the human annotations so we can compare their text with each new tweet."""
    data = annotations.reset_index(drop=True).copy()

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        stop_words="english",
        sublinear_tf=True,
        max_features=20_000,
    )

    matrix = vectorizer.fit_transform(
        data["Tweet"].fillna("").map(normalise_text)
    )

    return {
        "data": data,
        "vectorizer": vectorizer,
        "matrix": matrix,
    }


def select_similar_examples(
    tweet,
    reference_bank,
    exclude_key=None,
    number_of_examples=NUMBER_OF_EXAMPLES,
):
    """Pick the requested number of human examples that are closest to the new tweet."""
    data = reference_bank["data"]
    vectorizer = reference_bank["vectorizer"]
    matrix = reference_bank["matrix"]

    query_vector = vectorizer.transform([normalise_text(tweet)])
    similarities = cosine_similarity(query_vector, matrix).ravel()

    ranked = data.copy()
    ranked["_similarity"] = similarities

    # Do not let a tweet use itself as an example.
    if exclude_key:
        ranked = ranked.loc[ranked["annotation_key"] != exclude_key]

    ranked = ranked.sort_values(
        ["_similarity", "annotation_id"],
        ascending=[False, True],
        kind="mergesort",
    )

    number_of_examples = min(number_of_examples, len(ranked))
    selected = ranked.head(number_of_examples).copy()

    # Keep a mix of examples rather than selecting six nearly identical cases.
    selected_max = selected[SCORE_COLUMNS].max(axis=1)

    if not (selected_max == 0).any():
        zero_candidates = ranked.loc[
            ranked[SCORE_COLUMNS].max(axis=1).eq(0)
            & ~ranked["annotation_id"].isin(selected["annotation_id"])
        ]
        if not zero_candidates.empty and len(selected) > 0:
            selected = pd.concat(
                [selected.iloc[:-1], zero_candidates.head(1)],
                ignore_index=True,
            )

    selected_max = selected[SCORE_COLUMNS].max(axis=1)

    if not (selected_max >= 80).any():
        high_candidates = ranked.loc[
            ranked[SCORE_COLUMNS].max(axis=1).ge(80)
            & ~ranked["annotation_id"].isin(selected["annotation_id"])
        ]
        if not high_candidates.empty and len(selected) > 0:
            selected = pd.concat(
                [selected.iloc[:-1], high_candidates.head(1)],
                ignore_index=True,
            )

    selected = selected.drop_duplicates("annotation_id").head(number_of_examples)

    if EXAMPLE_ORDER == "least_to_most_similar":
        selected = selected.sort_values(
            ["_similarity", "annotation_id"], ascending=[True, True]
        )
    elif EXAMPLE_ORDER == "most_to_least_similar":
        selected = selected.sort_values(
            ["_similarity", "annotation_id"], ascending=[False, True]
        )
    elif EXAMPLE_ORDER == "annotation_id":
        selected = selected.sort_values("annotation_id")
    else:
        raise ValueError("Invalid EXAMPLE_ORDER setting.")

    return selected.reset_index(drop=True)


def detail_line(row, prefix):
    """Put the extra annotation details onto one readable line."""
    severity = row.get(f"{prefix}_severity")
    role = row.get(f"{prefix}_role")
    status = row.get(f"{prefix}_status")
    reason = row.get(f"{prefix}_reason")

    values = [severity, role, status, reason]
    if all(pd.isna(value) or str(value).strip() == "" for value in values):
        return None

    return (
        f"severity={severity}; role={role}; "
        f"status={status}; reason={reason}"
    )


def format_examples(examples):
    """Turn the selected human examples into the text shown to the model."""
    if examples.empty:
        return "No demonstrations are provided for this run."

    blocks = []

    for position, (_, row) in enumerate(examples.iterrows(), start=1):
        human_output = {
            "trade_score": int(row["trade_score"]),
            "sanctions_score": int(row["sanctions_score"]),
            "fed_pressure_score": int(row["fed_pressure_score"]),
            "reasoning": str(row["human_reason"]),
        }

        details = []
        for prefix, label in [
            ("trade", "Trade"),
            ("sanctions", "Sanctions"),
            ("fed", "Federal Reserve"),
        ]:
            detail = detail_line(row, prefix)
            if detail:
                details.append(f"- {label}: {detail}")

        lines = [
            f"DEMONSTRATION {position} [{row['annotation_id']}]",
            f"Date: {row['Date']}",
            f"Tweet: {row['Tweet']}",
            "Human reference output:",
            json.dumps(human_output, ensure_ascii=False),
        ]

        if details:
            lines.extend(["Additional human annotation detail:", *details])

        blocks.append("\n".join(lines))

    return "\n\n".join(blocks)


# For the final run, all 100 human annotations are available as possible examples.
all_annotations_reference = build_reference_bank(master_annotations)


preview = select_similar_examples(
    "The Fed has raised rates too quickly and should cut them now.",
    all_annotations_reference,
)

display(
    preview[
        [
            "annotation_id",
            "Tweet",
            "trade_score",
            "sanctions_score",
            "fed_pressure_score",
            "_similarity",
        ]
    ]
)


In [ ]:
# 7. The scoring rules and the format we want back

RUBRIC = """
You score three independent indices for each Donald Trump tweet.

A. TRADE HOSTILITY INDEX
Measures hostility, tension, confrontation, pressure, threats, restrictions, or escalation
in US trade relations with another country or group of countries.

B. SANCTIONS THREAT INDEX
Measures the use, continuation, strengthening, removal, announcement, or threat of US
economic sanctions or comparable economic punishment against a foreign target.

C. FEDERAL RESERVE PRESSURE INDEX
Measures Trump's criticism of, pressure on, blame directed at, or attempted influence over
the Federal Reserve and its monetary-policy decisions. "Federal" government funding is not
the Federal Reserve.

For each index, assess:
1. Relevance: is the signal genuinely present?
2. Severity: Very Low, Low, Moderate, Severe, or Very Severe.
3. Role: Direct if Trump/the US drives the action or pressure; Indirect if mainly reporting,
   describing, or reacting.
4. Status: Concluded, Partially Concluded, Ongoing, or Imminent.
5. Final integer score.

Score ranges:
- 0: no relevant signal
- 1-19: Very Low
- 20-39: Low
- 40-59: Moderate
- 60-79: Severe
- 80-100: Very Severe

Within a severity range, Direct generally scores above Indirect, and Imminent/Ongoing
generally scores above Partially Concluded/Concluded, all else equal.

Important distinctions:
- Generic geopolitical hostility is not automatically trade hostility.
- Trade hostility is not automatically a sanctions threat.
- "Fed", "federal", or federal payments are not Federal Reserve pressure unless the tweet
  refers to the Federal Reserve, monetary policy, interest rates, Powell, or Fed decisions.
- An existing or completed sanctions action can still receive a non-zero sanctions score.
- Score each index independently. Multiple indices may be non-zero.
- Use only information contained in the tweet and its date.
- The human demonstrations are scoring anchors, not facts to copy mechanically.
"""


class TweetRiskScores(BaseModel):
    trade_score: int = Field(
        ge=0,
        le=100,
        description="Trade Hostility Index",
    )
    sanctions_score: int = Field(
        ge=0,
        le=100,
        description="Sanctions Threat Index",
    )
    fed_pressure_score: int = Field(
        ge=0,
        le=100,
        description="Federal Reserve Pressure Index",
    )
    reasoning: str = Field(
        min_length=10,
        max_length=1200,
        description=(
            "A concise audit explanation identifying the evidence for each non-zero score "
            "and explaining why irrelevant indices are zero. Do not provide hidden chain-of-thought."
        ),
    )


output_parser = PydanticOutputParser(pydantic_object=TweetRiskScores)

SYSTEM_PROMPT = """
You are a geopolitical-risk annotation model supporting an academic dissertation.

Apply the rubric consistently. Use the human demonstrations to calibrate the scale and the
boundaries between zero, low, moderate, severe and very severe scores. Do not average the
examples and do not copy a score merely because the same country or keyword appears.

<RUBRIC>
{rubric}
</RUBRIC>

<RETRIEVED_HUMAN_ANNOTATIONS>
{few_shot_examples}
</RETRIEVED_HUMAN_ANNOTATIONS>

Return only the requested structured result. Do not include markdown or <think> tags.
{format_instructions}
"""

USER_PROMPT = """
Score this target tweet independently.

Tweet ID: {tweet_id}
Date: {date}
Tweet: {tweet}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", USER_PROMPT),
    ]
).partial(
    rubric=RUBRIC,
    format_instructions=output_parser.get_format_instructions(),
)

print("Prompt version:", PROMPT_VERSION)


In [ ]:
# 8. Connect to Ollama and prepare the prompt

def check_ollama():
    """Check that Ollama is running before we start a long job."""
    import urllib.request

    base_url = OLLAMA_URL.rstrip("/")

    try:
        with urllib.request.urlopen(
            f"{base_url}/api/version", timeout=5
        ) as response:
            version = json.loads(response.read().decode("utf-8"))
    except Exception as error:
        raise RuntimeError(
            f"Cannot reach Ollama at {base_url}. "
            "Start the Ollama server, then run this cell again."
        ) from error

    try:
        with urllib.request.urlopen(
            f"{base_url}/api/tags", timeout=10
        ) as response:
            model_data = json.loads(response.read().decode("utf-8"))

        installed_models = [
            model.get("name", "")
            for model in model_data.get("models", [])
        ]

        if MODEL_NAME not in installed_models:
            print(
                f"WARNING: {MODEL_NAME!r} was not found. "
                f"Installed models: {installed_models}"
            )
    except Exception as error:
        print("WARNING: Could not read the installed-model list:", error)

    print("Ollama version:", version)
    return version


ollama_version = check_ollama()

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_URL,
    temperature=TEMPERATURE,
    num_ctx=NUM_CTX,
    num_predict=NUM_PREDICT,
    keep_alive=KEEP_ALIVE,
)


def extract_json_from_model(message):
    """
    This is the backup method for DeepSeek.

    It removes the <think> section and keeps the JSON answer that comes after it.
    """
    content = message.content if hasattr(message, "content") else str(message)
    content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()

    first_brace = content.find("{")
    last_brace = content.rfind("}")

    if first_brace == -1 or last_brace <= first_brace:
        raise ValueError(
            "No JSON object found in model output: " + content[:500]
        )

    return content[first_brace:last_brace + 1]


# Try the clean structured-output method first.
try:
    structured_llm = llm.with_structured_output(
        TweetRiskScores,
        method="json_schema",
    )
    structured_method = "native_structured_output"
except TypeError:
    structured_llm = llm.with_structured_output(TweetRiskScores)
    structured_method = "native_structured_output"

structured_chain = prompt | structured_llm

# If DeepSeek does not follow that format, use the fallback that already works.
fallback_chain = (
    prompt
    | llm
    | RunnableLambda(extract_json_from_model)
    | output_parser
)

print("Primary method:", structured_method)


In [ ]:
# 9. Score a single tweet

def convert_result_to_dict(result):
    """Convert either type of model result into a normal Python dictionary."""
    if isinstance(result, dict):
        return result
    if hasattr(result, "model_dump"):
        return result.model_dump()
    if hasattr(result, "dict"):
        return result.dict()
    raise TypeError(f"Unsupported model result: {type(result)}")


def score_one_tweet(
    tweet_id,
    date,
    tweet,
    reference_bank,
    exclude_key=None,
    number_of_examples=NUMBER_OF_EXAMPLES,
):
    """Find the examples, ask the model for the scores and check the result."""

    if number_of_examples == 0:
        examples = pd.DataFrame()
    else:
        examples = select_similar_examples(
            tweet=tweet,
            reference_bank=reference_bank,
            exclude_key=exclude_key,
            number_of_examples=number_of_examples,
        )

    prompt_values = {
        "tweet_id": str(tweet_id),
        "date": str(date),
        "tweet": str(tweet),
        "few_shot_examples": format_examples(examples),
    }

    primary_error = None

    try:
        model_result = structured_chain.invoke(prompt_values)
        method_used = structured_method
    except Exception as error:
        primary_error = repr(error)
        model_result = fallback_chain.invoke(prompt_values)
        method_used = "parser_fallback"

    result = TweetRiskScores.model_validate(
        convert_result_to_dict(model_result)
    ).model_dump()

    if primary_error:
        result["_primary_error"] = primary_error

    example_ids = (
        examples["annotation_id"].tolist()
        if not examples.empty
        else []
    )

    return result, example_ids, method_used


def score_with_retries(**arguments):
    """Try again a few times if Ollama has a temporary problem."""
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return score_one_tweet(**arguments)
        except Exception as error:
            last_error = error

            if attempt == MAX_RETRIES:
                break

            wait_seconds = min(2 ** (attempt - 1), 8)
            print(
                f"Attempt {attempt} failed: {error}. "
                f"Retrying in {wait_seconds} seconds."
            )
            time.sleep(wait_seconds)

    raise RuntimeError(
        f"Scoring failed after {MAX_RETRIES} attempts: {last_error}"
    ) from last_error


## 10. Optional test on one tweet

Set `RUN_SINGLE_TWEET_TEST = True` when you only want to check one model response.

I have left it as `False` by default, so **Run All** goes straight to the full dataset.


In [ ]:
RUN_SINGLE_TWEET_TEST = False

if RUN_SINGLE_TWEET_TEST:
    example_tweet = (
        "Many people talking, with much agreement, on my Iran speech today. "
        "Participants in the deal are making lots of money on trade with Iran!"
    )

    test_result, test_examples, test_method = score_with_retries(
        tweet_id="919005534883328e3",
        date="2017-10-14 01:02:48",
        tweet=example_tweet,
        reference_bank=all_annotations_reference,
        exclude_key=make_annotation_key(
            "2017-10-14 01:02:48", example_tweet
        ),
        number_of_examples=NUMBER_OF_EXAMPLES,
    )

    print("Method:", test_method)
    print("Examples:", test_examples)
    print(json.dumps(test_result, indent=2, ensure_ascii=False))


In [ ]:
# 11. Save each result safely as soon as it is ready

OUTPUT_COLUMNS = [
    "tweet_index",
    "tweet_id",
    "date",
    "tweet_snippet",
    "trade_score",
    "sanctions_score",
    "fed_pressure_score",
    "reasoning",
    "source",
    "example_ids",
    "model",
    "prompt_version",
    "created_at_utc",
]

ERROR_COLUMNS = [
    "tweet_index",
    "tweet_id",
    "date",
    "tweet_snippet",
    "error",
    "model",
    "prompt_version",
    "created_at_utc",
]


def make_snippet(tweet, max_characters=TWEET_SNIPPET_LENGTH):
    """Make the short tweet preview that goes into the CSV."""
    text = re.sub(r"\s+", " ", str(tweet)).strip()

    if len(text) <= max_characters:
        return text

    return text[:max_characters - 1].rstrip() + "…"


def append_row_to_csv(path, row, columns):
    """Add one result to the CSV straight away and make sure it is saved."""
    path.parent.mkdir(parents=True, exist_ok=True)
    needs_header = not path.exists() or path.stat().st_size == 0

    with path.open("a", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=columns,
            extrasaction="ignore",
        )

        if needs_header:
            writer.writeheader()

        writer.writerow({column: row.get(column) for column in columns})

        # This makes sure the row is actually written before we move on.
        file.flush()
        os.fsync(file.fileno())


def load_completed_indices(path):
    """Find the tweets that have already been completed in this CSV."""
    completed = set()

    if not path.exists() or path.stat().st_size == 0:
        return completed

    try:
        with path.open("r", encoding="utf-8", newline="") as file:
            for row in csv.DictReader(file):
                try:
                    tweet_index = int(row.get("tweet_index", ""))

                    has_all_scores = all(
                        str(row.get(column, "")).strip() != ""
                        for column in SCORE_COLUMNS
                    )

                    if has_all_scores:
                        completed.add(tweet_index)

                except (TypeError, ValueError):
                    # A crash can leave one unfinished line at the end, so just ignore that line.
                    continue

    except csv.Error as error:
        print("WARNING: CSV reading stopped at a malformed line:", error)

    return completed


In [ ]:
# 12. Run the full dataset and continue from where it stopped

def package_versions():
    """Record the package versions so we know exactly what was used."""
    names = [
        "pandas",
        "numpy",
        "scikit-learn",
        "pydantic",
        "langchain",
        "langchain-core",
        "langchain-ollama",
    ]

    versions = {}

    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None

    return versions


def save_run_manifest(number_of_requested_rows):
    """Save the setup used for this experiment."""
    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": MODEL_NAME,
        "ollama_base_url": OLLAMA_URL,
        "prompt_version": PROMPT_VERSION,
        "temperature": TEMPERATURE,
        "num_ctx": NUM_CTX,
        "num_predict": NUM_PREDICT,
        "keep_alive": KEEP_ALIVE,
        "seed": SEED,
        "n_retrieved_human_examples": NUMBER_OF_EXAMPLES,
        "example_order": EXAMPLE_ORDER,
        "preserve_human_labels": PRESERVE_HUMAN_LABELS,
        "final_reference_bank": "all human annotations",
        "file_hashes_sha256": {
            name: file_sha256(path)
            for name, path in PATHS.items()
        },
        "package_versions": package_versions(),
        "live_csv": str(LIVE_CSV),
        "error_csv": str(ERROR_CSV),
        "rows_requested_this_call": number_of_requested_rows,
        "successful_indices_after_call": len(
            load_completed_indices(LIVE_CSV)
        ),
    }

    MANIFEST_FILE.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def run_production_scoring(
    tweet_data,
    max_tweets=MAX_TWEETS,
    preserve_human_labels=PRESERVE_HUMAN_LABELS,
):
    """Score every tweet that is not finished yet and save each result immediately."""

    rows = tweet_data.copy()

    if max_tweets is not None:
        rows = rows.head(max_tweets).copy()

    completed_indices = load_completed_indices(LIVE_CSV)

    human_lookup = {
        row["annotation_key"]: row
        for row in master_annotations.to_dict(orient="records")
    }

    print(f"Rows requested: {len(rows):,}")
    print(f"Already completed: {len(completed_indices):,}")
    print("Successful results:", LIVE_CSV.resolve())
    print("Failed results:", ERROR_CSV.resolve())

    for _, row in tqdm(
        rows.iterrows(),
        total=len(rows),
        desc="Scoring tweets",
    ):
        tweet_index = int(row["tweet_index"])

        # If this tweet is already in the CSV, there is no need to run it again.
        if tweet_index in completed_indices:
            continue

        annotation_key = row["annotation_key"]
        created_at = datetime.now(timezone.utc).isoformat()

        shared_output = {
            "tweet_index": tweet_index,
            "tweet_id": row.get("Tweet ID"),
            "date": row.get("Date"),
            "tweet_snippet": make_snippet(row.get("Tweet")),
            "model": MODEL_NAME,
            "prompt_version": PROMPT_VERSION,
            "created_at_utc": created_at,
        }

        try:
            # If we already scored this tweet by hand, keep the human scores.
            if preserve_human_labels and annotation_key in human_lookup:
                human_row = human_lookup[annotation_key]

                result_row = {
                    **shared_output,
                    "trade_score": int(human_row["trade_score"]),
                    "sanctions_score": int(human_row["sanctions_score"]),
                    "fed_pressure_score": int(
                        human_row["fed_pressure_score"]
                    ),
                    "reasoning": str(human_row["human_reason"]),
                    "source": "human_annotation",
                    "example_ids": "",
                }

            # Otherwise, ask the model and show it the selected human examples first.
            else:
                result, example_ids, method = score_with_retries(
                    tweet_id=row.get("Tweet ID"),
                    date=row.get("Date"),
                    tweet=row.get("Tweet"),
                    reference_bank=all_annotations_reference,
                    exclude_key=annotation_key,
                    number_of_examples=NUMBER_OF_EXAMPLES,
                )

                result_row = {
                    **shared_output,
                    "trade_score": int(result["trade_score"]),
                    "sanctions_score": int(result["sanctions_score"]),
                    "fed_pressure_score": int(
                        result["fed_pressure_score"]
                    ),
                    "reasoning": str(result["reasoning"]),
                    "source": f"llm:{method}",
                    "example_ids": "|".join(example_ids),
                }

            # Save this result before starting the next tweet.
            append_row_to_csv(
                LIVE_CSV,
                result_row,
                OUTPUT_COLUMNS,
            )
            completed_indices.add(tweet_index)

        except Exception as error:
            error_row = {
                **shared_output,
                "error": repr(error),
            }

            append_row_to_csv(
                ERROR_CSV,
                error_row,
                ERROR_COLUMNS,
            )

            print(
                f"Tweet index {tweet_index} failed and "
                f"was logged for retry: {error}"
            )

    save_run_manifest(len(rows))

    if LIVE_CSV.exists():
        output = pd.read_csv(LIVE_CSV)
        output = (
            output
            .drop_duplicates("tweet_index", keep="last")
            .sort_values("tweet_index")
            .reset_index(drop=True)
        )
    else:
        output = pd.DataFrame(columns=OUTPUT_COLUMNS)

    return output


# Running this cell starts the full run, or continues the existing one.
annotation_output = run_production_scoring(tweets)

display(annotation_output.tail())
print("Crash-safe live CSV:", LIVE_CSV.resolve())


In [ ]:
# 13. Final checks that do not use the GPU or call the model

assert len(master_annotations) == 100
assert master_annotations["annotation_key"].is_unique
assert tweets["annotation_key"].is_unique
assert tweets["tweet_index"].tolist() == list(range(len(tweets)))

for column in SCORE_COLUMNS:
    assert master_annotations[column].between(0, 100).all()

print("Local data and retrieval checks passed.")
print("Live output CSV:", LIVE_CSV.resolve())
print("Error CSV:", ERROR_CSV.resolve())
